In [1]:
load_ext jupyter_black

In [2]:
import os
import pandas as pd
from scipy.stats import f_oneway, tukey_hsd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

In [3]:
demographics = {
    "prism": [
        "age",
        "gender",
        "employment_status",
        "education",
        "marital_status",
        "english_proficiency",
        "religion",
        "ethnicity",
        "birth_region",
        "reside_region",
        "lm_familiarity",
    ],
    "chen": [
        "label",
        "human_Gender",
    ],
    "cad_en": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_fr": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_pt": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
    "cad_it": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
}

id_col = {
    "prism": "conversation_id",
    "chen": "text_id",
    "cad_en": "conversation_id",
    "cad_fr": "conversation_id",
    "cad_pt": "conversation_id",
    "cad_it": "conversation_id",
}

In [4]:
for dataset in ["cad_fr", "cad_pt", "cad_it", "chen", "prism", "cad_en"]:
    if not os.path.exists(f"figures_{dataset}"):
        os.makedirs(f"figures_{dataset}")
    df = pd.read_pickle(
        f"data/{dataset + '_utterances' if dataset != 'chen' else dataset}_linguistic.gz"
    )
    for c in ["politeness_user_prompt", "politeness_model_response"]:
        if c in df:
            df[c] = df[c].replace(
                {"impolite": -2, "neutral": 0, "polite": 2, "somewhat polite": 1}
            )
    df = df.rename(columns={"gpt_description": "topic"})
    if dataset != "chen":
        if dataset == "prism":
            demographics[dataset] += ["model_name"]
            demographics[dataset] += ["topic"]
        group_cols = [id_col[dataset]] + demographics[dataset]
        df = (
            df.groupby(group_cols)[
                [c for c in df.columns if "_model_response" in c or "_user_prompt" in c]
            ]
            .mean()
            .reset_index()
        )

    if dataset in ["chen", "prism", "cad_en"]:
        df_beliefs = pd.read_pickle(
            f"llama_beliefs/Llama-3.1-8B-Instruct_{dataset}_beliefs_preprocessed.gz"
        )
        cols = [
            c
            for c in df_beliefs.columns
            if "shared_extracted_" in c or "value_JSON_" in c or "unknown_token_" in c
        ] + ["revealed_Gender"]
        if "human_Gender" in df_beliefs.columns:
            cols += ["human_Gender"]
        demographics[dataset] += cols
        cols.append(id_col[dataset])
        df = df.merge(df_beliefs[cols], on=id_col[dataset])
    num_cols = [c for c in df.columns if "_model_response" in c or "_user_prompt" in c]
    # x-axis compared to y-axis
    for demographic in tqdm(demographics[dataset]):
        filtered_df = (
            df.loc[~df[demographic].isna()]
            .groupby(demographic)
            .filter(lambda x: len(x) > 1)
        )
        for num_col in tqdm(num_cols):
            if os.path.isfile(
                f"figures_{dataset}/llama_{dataset}_{demographic.replace(' ','')}_{num_col.replace(' ','')}.png"
            ):
                continue
            if (
                f_oneway(
                    *filtered_df.groupby(demographic)[num_col].apply(list).tolist()
                ).pvalue
                < 0.05
            ):
                t = tukey_hsd(
                    *filtered_df.groupby(demographic)[num_col].apply(list).tolist()
                )
                if (t.pvalue >= 0.05).all():
                    continue
                groups = sorted(filtered_df[demographic].unique())
                means = filtered_df.groupby(demographic)[num_col].mean().values
                differences = means - means[:, None]
                ax = sns.heatmap(differences, mask=t.pvalue >= 0.05, annot=True)
                ax.set_xticklabels(groups)
                ax.set_yticklabels(groups)
                ax.tick_params(axis="both", which="major", labelsize=6)
                plt.title(f"{demographic} - {num_col}")
                f = ax.get_figure()
                f.savefig(
                    f"figures_{dataset}/llama_{dataset}_{demographic.replace(' ','')}_{num_col.replace(' ','')}.png"
                )
                f.clear()
                plt.close(f)

  0%|                                                                                                                                                                                                                 | 0/5 [00:00<?, ?it/s]
%|                                                                                                                                                                                                                | 0/76 [00:00<?, ?it/s]
%|███████████████████████▋                                                                                                                                                                                | 9/76 [00:00<00:04, 16.03it/s]
%|████████████████████████████▊                                                                                                                                                                          | 11/76 [00:00<00:05, 11.31it/s]
%|█████████████████████████████████████████▉                 